# Speaker Encoder & d-Vector

## What is a Speaker Embedding?

When we want a TTS model to speak in a specific person's voice, we need a way to **represent that voice** as a fixed-size vector.

A **speaker encoder** is a neural network trained to map any audio utterance from a person into a compact vector (called a **d-vector**) such that:
- Utterances from the **same speaker** are **close** in embedding space
- Utterances from **different speakers** are **far apart**

This vector captures voice characteristics like timbre, pitch range, and speaking style — independent of the spoken content.

```
"Hello world" (Speaker A) --> [0.12, -0.33, 0.87, ...]  --|
"How are you?" (Speaker A) --> [0.14, -0.30, 0.85, ...]  --|--> cluster together
"Good morning" (Speaker A) --> [0.11, -0.35, 0.89, ...]  --|

"Hello world" (Speaker B) --> [-0.55, 0.71, -0.22, ...] --> far from Speaker A
```

The key property: the encoder **generalizes to unseen speakers** — it never saw the target speaker during training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Architecture: LSTM-based Speaker Encoder

**Paper:** [Transfer Learning from Speaker Verification to Multispeaker TTS](https://arxiv.org/abs/1806.04558) (Wan et al., 2018)

The speaker encoder uses a **3-layer LSTM** that processes mel spectrogram frames.

Why LSTM?
- Audio utterances have **variable length** — LSTM handles this naturally
- The final hidden state aggregates information from the entire utterance

Architecture:
```
Mel Spectrogram (T x n_mels)
        |
   LSTM x 3 layers
        |
   Last hidden state  (d_hidden)
        |
   Linear projection
        |
   L2 Normalize
        |
   d-vector  (d_embed)   <- unit norm, lives on hypersphere
```

In [ ]:
class SpeakerEncoder(nn.Module):
    def __init__(self, n_mels=40, d_hidden=256, n_layers=3, d_embed=256):
        super().__init__()
        self.lstm = nn.LSTM(n_mels, d_hidden, num_layers=n_layers, batch_first=True)
        self.proj = nn.Linear(d_hidden, d_embed)

    def forward(self, mel):
        # mel: (B, T, n_mels)
        _, (h_n, _) = self.lstm(mel)     # h_n: (n_layers, B, d_hidden)
        h = h_n[-1]                       # last layer: (B, d_hidden)
        embed = self.proj(h)
        embed = F.normalize(embed, dim=-1)   # L2 norm -> unit hypersphere
        return embed

spk_enc = SpeakerEncoder(n_mels=40, d_hidden=128, n_layers=3, d_embed=128)

# Test: 4 speakers x 10 utterances each (variable length audio)
n_speakers, n_utterances = 4, 10
mel_batch = torch.randn(n_speakers * n_utterances, 160, 40)   # 160 frames = ~1 sec
embeds = spk_enc(mel_batch)
embeds = embeds.view(n_speakers, n_utterances, -1)
print("Embedding shape:", embeds.shape)
print("L2 norm (should be 1.0):", embeds[0, 0].norm().item())

## 2. GE2E Loss (Generalized End-to-End Loss)

**Paper:** [Generalized End-to-End Loss for Speaker Verification](https://arxiv.org/abs/1710.10467) (Wan et al., 2018)

Standard classification loss doesn't work here because:
- The number of speakers in training is fixed, but we want to generalize to **any new speaker**
- We want to learn a **metric space**, not classify speakers by ID

**GE2E loss** computes a similarity matrix between all utterances and all speaker centroids:

```
Centroid c_k = mean of all embeddings from speaker k

Similarity S[i,j,k] = cos_sim( embed[i,j] , centroid_k )

Loss = -log( softmax(S[i,j,:])[own_speaker] )
```

This pulls same-speaker embeddings together and pushes different-speaker embeddings apart.

In [ ]:
def ge2e_loss(embeds, w=10.0, b=-5.0):
    # embeds: (N_speakers, N_utterances, d_embed) — already L2 normalized
    N, M, d = embeds.shape

    centroids = F.normalize(embeds.mean(dim=1), dim=-1)   # (N, d)

    sim = torch.zeros(N, M, N)
    for i in range(N):
        for j in range(M):
            for k in range(N):
                if i == k:
                    # Leave-one-out centroid for positive pair (avoids trivial solution)
                    c = F.normalize((embeds[i].sum(0) - embeds[i, j]) / (M - 1), dim=0)
                else:
                    c = centroids[k]
                sim[i, j, k] = torch.dot(embeds[i, j], c)

    sim = w * sim + b   # learnable scale and bias

    # Each utterance (i,j) should be most similar to centroid k=i
    labels = torch.arange(N).unsqueeze(1).expand(N, M).reshape(-1)
    loss = F.cross_entropy(sim.reshape(N * M, N), labels)
    return loss

embeds_leaf = embeds.detach().clone().requires_grad_(True)
loss = ge2e_loss(embeds_leaf)
print(f"GE2E loss: {loss.item():.4f}")
print("Lower loss = better speaker discrimination")

## 3. Speaker Similarity Matrix

After training, we expect:
- **High similarity** (bright) along the diagonal blocks (same speaker)
- **Low similarity** (dark) in off-diagonal blocks (different speakers)

In [ ]:
# Simulate well-trained embeddings with visible clusters
cluster_centers = F.normalize(torch.randn(n_speakers, 128), dim=-1)
structured = []
for i in range(n_speakers):
    noise = torch.randn(n_utterances, 128) * 0.05
    utts  = cluster_centers[i].unsqueeze(0) + noise
    utts  = F.normalize(utts, dim=-1)
    structured.append(utts)
structured = torch.cat(structured, dim=0)   # (40, 128)

sim_matrix = (structured @ structured.T).detach().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Speaker Embedding Cosine Similarity\n(4 speakers x 10 utterances each)")
ax.set_xlabel("Utterance index")
ax.set_ylabel("Utterance index")
for i in [10, 20, 30]:
    ax.axhline(i - 0.5, color="k", lw=2)
    ax.axvline(i - 0.5, color="k", lw=2)
ax.set_xticks([5, 15, 25, 35])
ax.set_xticklabels(["Spk 1", "Spk 2", "Spk 3", "Spk 4"])
ax.set_yticks([5, 15, 25, 35])
ax.set_yticklabels(["Spk 1", "Spk 2", "Spk 3", "Spk 4"])
plt.tight_layout()
plt.savefig("figures/speaker_sim.png", dpi=110, bbox_inches="tight")
plt.show()
print("Diagonal blocks = same speaker -> should be bright (high similarity)")

## 4. t-SNE Visualization of Speaker Embeddings

In [ ]:
from sklearn.manifold import TSNE

n_spk, n_utt = 6, 20
centers = F.normalize(torch.randn(n_spk, 128), dim=-1)
all_embeds = []
for i in range(n_spk):
    noise = torch.randn(n_utt, 128) * 0.04
    utts  = F.normalize(centers[i] + noise, dim=-1)
    all_embeds.append(utts)
all_embeds = torch.cat(all_embeds, dim=0).numpy()

tsne   = TSNE(n_components=2, perplexity=20, random_state=42)
coords = tsne.fit_transform(all_embeds)

fig, ax = plt.subplots(figsize=(7, 6))
colors = plt.cm.tab10.colors
for i in range(n_spk):
    idx = slice(i * n_utt, (i+1) * n_utt)
    ax.scatter(coords[idx, 0], coords[idx, 1],
               c=[colors[i]], label=f"Speaker {i+1}", alpha=0.85, s=60)
ax.legend(loc="best", fontsize=9)
ax.set_title("t-SNE of Speaker Embeddings\n(6 speakers x 20 utterances each)")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/tsne_speakers.png", dpi=110, bbox_inches="tight")
plt.show()
print("Good speaker encoder: each speaker forms a tight, well-separated cluster")

## Summary

| Concept | Detail |
|---------|--------|
| **Speaker Encoder** | 3-layer LSTM -> Linear -> L2 normalize |
| **d-vector** | Fixed-size (256-d) speaker embedding on unit hypersphere |
| **Training task** | Speaker verification (are these two utterances from the same person?) |
| **GE2E Loss** | Similarity matrix loss — pulls same-speaker together, pushes different apart |
| **Key property** | Generalizes to **unseen speakers** at inference time |

**Next:** SV2TTS — inject the d-vector into a Tacotron 2 / FastSpeech 2 model to clone any voice.